# تست تشخیصی نسخه‌ی ۳: رفع خطای مرزی در گرد کردن fractional (0.000 vs 0.999)

## چرا این نسخه؟
نسخه‌ی ۲ (تطبیق بر اساس بردار کامل) در ۱۰ از ۱۱ ماده با خطای **"عدم تطابق کلید (کم=1,
زیاد=1)"** شکست خورد — یعنی همیشه دقیقاً یک کلید با یکی دیگر فرق دارد، نه آشفتگی کامل.

این الگو امضای کلاسیک **خطای مرزی گرد کردن** است: مقداری مثل `0.99999970` در یک گروه ممکن
است به `1.000` گرد شود، در حالی که در گروه دیگر (با خطای عددی کمی متفاوت Phonopy) همان
مقدار فیزیکی به `0.000` گرد می‌شود. این دو از نظر فیزیکی کاملاً یکی هستند (چون fractional
به‌صورت دوره‌ای با تناوب ۱ تعریف می‌شود) ولی به‌عنوان کلید رشته‌ای متفاوت دیده می‌شوند.

نمونه‌ی موفق (`Cr2AsC`) هم مقدار `1.0000` را در خروجی نشان داد -- که خودش گواه این مشکل
مرزی است؛ فقط در آن مورد به‌طور تصادفی تناقضی رخ نداد.

## رفع باگ
بعد از گرد کردن، یک بار دیگر `np.mod(rounded, 1.0)` می‌گیریم تا هر مقداری که دقیقاً `1.000`
شده (به‌خاطر گرد کردن یک عدد نزدیک به مرز) به `0.000` نگاشت شود -- این یک wrap-around نهایی
ساده و قطعی است.

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
print(f"تعداد مواد یافت‌شده: {len(common)}")

test_formulas = common[:5] + common[len(common)//2:len(common)//2+3] + common[-3:]
test_formulas = list(dict.fromkeys(test_formulas))
print(f"مواد تستی ({len(test_formulas)}): {test_formulas}")

## تابع کشف Supercell (بدون تغییر)

In [ ]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc
    return {'phonon': phonon, 'supercell_dim': dim, 'dim_match_error': err}

print("✅ آماده شد")

## تابع اصلاح‌شده v3: wrap-around دوم بعد از گرد کردن (رفع خطای مرزی 0.000 vs 1.000)

In [ ]:
def extract_image_displacement_vectors_v3(phonon, n_atoms, n_images, round_decimals=3):
    """
    تطبیق بر اساس بردار کامل fractional، با رفع خطای مرزی گرد کردن (0.000 vs 1.000).
    """
    supercell = phonon.supercell
    sc_cart = supercell.positions
    sc_lattice = supercell.cell
    s2u_map = supercell.s2u_map

    try:
        inv_lattice = np.linalg.inv(sc_lattice)
    except np.linalg.LinAlgError:
        return None, "لاتیس singular"

    unique_reps = np.unique(s2u_map)
    if len(unique_reps) != n_atoms:
        return None, "تعداد گروه نامنطبق"

    groups = {rep: np.where(s2u_map == rep)[0] for rep in unique_reps}
    if not all(len(v) == n_images for v in groups.values()):
        return None, "اندازه‌ی گروه نامنطبق"

    per_group_frac = {}
    for rep in unique_reps:
        members = groups[rep]
        disp_cart = sc_cart[members] - sc_cart[rep]
        disp_frac = disp_cart @ inv_lattice
        disp_frac_wrapped = np.mod(disp_frac, 1.0)
        disp_frac_rounded = np.round(disp_frac_wrapped, decimals=round_decimals)
        # رفع خطای مرزی: بعد از گرد کردن، دوباره mod بگیریم تا 1.000 -> 0.000 شود
        disp_frac_rounded = np.mod(disp_frac_rounded, 1.0)
        # گرد کردن دوباره برای رفع خطای عددی باقی‌مانده از mod دوم (مثلاً 0.9999999999 -> 1.0 -> 0.0)
        disp_frac_rounded = np.round(disp_frac_rounded, decimals=round_decimals)
        per_group_frac[rep] = disp_frac_rounded

    first_rep = unique_reps[0]
    reference_keys = [tuple(v) for v in per_group_frac[first_rep]]

    for rep in unique_reps[1:]:
        this_keys = set(tuple(v) for v in per_group_frac[rep])
        ref_keys_set = set(reference_keys)
        if this_keys != ref_keys_set:
            missing = ref_keys_set - this_keys
            extra = this_keys - ref_keys_set
            return None, f"عدم تطابق کلید بین گروه‌ها (گروه {rep}, کم={missing}, زیاد={extra})"

    sorted_keys = sorted(set(reference_keys))
    if len(sorted_keys) != n_images:
        return None, f"تعداد کلید یکتا ({len(sorted_keys)}) با n_images ({n_images}) نمی‌خواند"

    final_vectors = np.array(sorted_keys, dtype=np.float32)
    return final_vectors, "OK"

print("✅ تابع v3 آماده شد")

## استخراج و چاپ بردارهای جابجایی (نسخه‌ی اصلاح‌شده v3) برای هر ماده‌ی تستی

In [ ]:
N_ATOMS_FIXED = 8
N_IMAGES_FIXED = 9

success_count = 0
fail_count = 0

for formula in test_formulas:
    print(f"\n{'='*60}")
    print(f"ماده: {formula}")
    try:
        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            print("  ❌ load_material_with_phonopy شکست خورد")
            fail_count += 1
            continue

        print(f"  supercell_dim کشف‌شده: {result['supercell_dim']} (خطا: {result['dim_match_error']:.6f})")

        disp_vectors, status = extract_image_displacement_vectors_v3(
            result['phonon'], N_ATOMS_FIXED, N_IMAGES_FIXED)

        if disp_vectors is None:
            print(f"  ❌ استخراج بردار جابجایی شکست خورد: {status}")
            fail_count += 1
            continue

        success_count += 1
        print(f"  ✅ بردارهای جابجایی fractional (shape={disp_vectors.shape}):")
        for k, v in enumerate(disp_vectors):
            print(f"     image {k}: [{v[0]:+.4f}, {v[1]:+.4f}, {v[2]:+.4f}]  (norm={np.linalg.norm(v):.4f})")

        all_norms = np.linalg.norm(disp_vectors, axis=1)
        print(f"  📊 norm: min={all_norms.min():.4f}, max={all_norms.max():.4f}, std={all_norms.std():.4f}")

        from scipy.spatial.distance import pdist
        pairwise_dists = pdist(disp_vectors)
        print(f"  📊 حداقل فاصله‌ی زوجی بین بردارها: {pairwise_dists.min():.6f}")

    except Exception as e:
        print(f"  ❌ خطای غیرمنتظره: {e}")
        fail_count += 1

print(f"\n{'='*60}")
print(f"📊 نتیجه‌ی کلی: {success_count} موفق، {fail_count} شکست (از {len(test_formulas)} ماده‌ی تستی)")

## نتیجه‌گیری این تست

اگر این‌بار همه (یا اکثر) مواد موفق شدند، یعنی مشکل واقعاً همان خطای مرزی گرد کردن بود و
حل شده است. اگر باز هم خطاهایی باقی ماندند، پیام دقیق خطا (که این‌بار به‌جای فقط شمارش،
خودِ کلیدهای کم/زیاد را چاپ می‌کند) به ما می‌گوید دقیقاً چه مقادیری با هم ناسازگارند، که
می‌تواند نشانه‌ی یک مشکل دیگر (نه صرفاً مرزی) باشد.

**لطفاً کل خروجی این تست را برای من بفرستید.**